In [1]:
import os, json, shutil
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score, classification_report

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA RTX A4000


In [2]:
ROOT = os.path.abspath(os.path.join('..', '..'))
BASE = os.path.join(ROOT, 'Roman', 'Product or eCommerce Reviews')

dfa_train = pd.read_csv(os.path.join(BASE, 'Domain_A_Daraz_Ecommerce', 'train.csv'), encoding='utf-8-sig')
dfa_test  = pd.read_csv(os.path.join(BASE, 'Domain_A_Daraz_Ecommerce', 'test.csv'),  encoding='utf-8-sig')
dfb_train = pd.read_csv(os.path.join(BASE, 'Domain_B_Restaurant_Food', 'train.csv'), encoding='utf-8-sig')
dfb_test  = pd.read_csv(os.path.join(BASE, 'Domain_B_Restaurant_Food', 'test.csv'),  encoding='utf-8-sig')
dfd_train = pd.read_csv(os.path.join(BASE, 'Domain_D_Electronics_Gaming_Delivery', 'train.csv'), encoding='utf-8-sig')
dfd_test  = pd.read_csv(os.path.join(BASE, 'Domain_D_Electronics_Gaming_Delivery', 'test.csv'),  encoding='utf-8-sig')

# Labels are text: Positive, Negative, Neutral -> map to integers
LABEL_MAP = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
ID2LABEL  = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}
LABEL2ID  = {'NEGATIVE': 0, 'NEUTRAL': 1, 'POSITIVE': 2}

for df in [dfa_train, dfa_test, dfb_train, dfb_test, dfd_train, dfd_test]:
    df['label'] = df['label'].map(LABEL_MAP)

print('Domain sizes (train / test):')
for name, tr, te in [
    ('A_Daraz_Ecommerce',             dfa_train, dfa_test),
    ('B_Restaurant_Food',             dfb_train, dfb_test),
    ('D_Electronics_Gaming_Delivery', dfd_train, dfd_test),
]:
    print(f'  {name}: train={len(tr)}  test={len(te)}  labels={tr["label"].value_counts().to_dict()}')

Domain sizes (train / test):
  A_Daraz_Ecommerce: train=13592  test=3398  labels={2: 8133, 0: 3490, 1: 1969}
  B_Restaurant_Food: train=600  test=150  labels={2: 473, 1: 93, 0: 34}
  D_Electronics_Gaming_Delivery: train=22471  test=5618  labels={2: 9036, 0: 8279, 1: 5156}


In [3]:
XLM_MODEL_ID  = 'xlm-roberta-base'
BERT_MODEL_ID = 'bert-base-multilingual-cased'

NUM_LABELS    = 3
MAX_LEN       = 128
MAX_TRAIN     = 8000
EPOCHS        = 5
PATIENCE      = 2
BATCH_TRAIN   = 16
BATCH_EVAL    = 32
LR            = 2e-5

RESULTS_BASE  = os.path.join(ROOT, 'results', 'T6_Roman_ProductReviews')
os.makedirs(RESULTS_BASE, exist_ok=True)

domains = [
    ('A_Daraz_Ecommerce',             dfa_train, dfa_test),
    ('B_Restaurant_Food',             dfb_train, dfb_test),
    ('D_Electronics_Gaming_Delivery', dfd_train, dfd_test),
]

In [4]:
def cap_dataset(df, max_samples=MAX_TRAIN):
    if len(df) <= max_samples:
        return df
    samples = []
    for label, group in df.groupby('label'):
        n = min(len(group), round(max_samples * len(group) / len(df)))
        samples.append(group.sample(n=n, random_state=42))
    capped = pd.concat(samples, ignore_index=True)
    return capped.sample(frac=1, random_state=42).reset_index(drop=True)


class UrduDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.labels = df['label'].astype(int).tolist()
        self.enc = tokenizer(
            df['text'].astype(str).tolist(),
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds)
    }


def run_one(model_id, model_label, src_name, train_df, tgt_name, test_df):
    out_dir     = os.path.join(RESULTS_BASE, model_label)
    os.makedirs(out_dir, exist_ok=True)
    result_path = os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')

    if os.path.exists(result_path):
        print(f'  [SKIP] {src_name} -> {tgt_name} already done.')
        with open(result_path, encoding='utf-8') as f:
            return json.load(f)['macro_f1'], None

    run_type = 'IN-DOMAIN' if src_name == tgt_name else 'CROSS-DOMAIN'
    capped   = cap_dataset(train_df)
    print(f'\n  [{run_type}] Train: {src_name} ({len(capped)}) -> Test: {tgt_name} ({len(test_df)})')

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_LABELS)

    val_df    = capped.sample(max(int(len(capped) * 0.1), 1), random_state=42)
    train_sub = capped.drop(val_df.index)

    ckpt_dir = os.path.join(out_dir, f'_ckpt_{src_name}_{tgt_name}')

    args = TrainingArguments(
        output_dir                  = ckpt_dir,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_TRAIN,
        per_device_eval_batch_size  = BATCH_EVAL,
        learning_rate               = LR,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'macro_f1',
        greater_is_better           = True,
        logging_steps               = 50,
        fp16                        = torch.cuda.is_available(),
        report_to                   = 'none',
        save_total_limit            = 1,
        save_only_model              = True,  # skip optimizer/scheduler state to save disk space
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = UrduDataset(train_sub, tokenizer),
        eval_dataset    = UrduDataset(val_df, tokenizer),
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    )
    trainer.train()

    preds_out = trainer.predict(UrduDataset(test_df, tokenizer))
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = preds_out.label_ids

    macro_f1 = f1_score(labels, preds, average='macro')
    accuracy = accuracy_score(labels, preds)

    result = {
        'task': 'T6_Roman_ProductReviews', 'model': model_label, 'model_id': model_id,
        'source': src_name, 'target': tgt_name, 'type': run_type,
        'train_size': len(capped), 'test_size': len(test_df),
        'macro_f1': round(macro_f1, 4), 'accuracy': round(accuracy, 4),
        'classification_report': classification_report(labels, preds, output_dict=True)
    }
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    if os.path.exists(ckpt_dir):
        shutil.rmtree(ckpt_dir)

    print(f'  macro-F1={macro_f1:.4f}  accuracy={accuracy:.4f}')
    return macro_f1, trainer


print('Helpers loaded. Ready to run experiments.')

Helpers loaded. Ready to run experiments.


## Model 1 — XLM-R Base

In [5]:
print('=== XLM-R | Source: A_Daraz_Ecommerce ===')
xlmr_results = globals().get('xlmr_results', {})

src_name, train_df, _ = domains[0]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource A done.')

=== XLM-R | Source: A_Daraz_Ecommerce ===

  [IN-DOMAIN] Train: A_Daraz_Ecommerce (8000) -> Test: A_Daraz_Ecommerce (3398)


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maryam\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14067.66it/s]
[transformers] 

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.449758,0.318591,0.844048,0.895000
2,0.323406,0.284804,0.859441,0.910000
3,0.280628,0.319266,0.851175,0.908750
4,0.286673,0.415155,0.843389,0.900000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.73it/s]


  macro-F1=0.8631  accuracy=0.9049

  [CROSS-DOMAIN] Train: A_Daraz_Ecommerce (8000) -> Test: B_Restaurant_Food (150)


...

  macro-F1=0.5191  accuracy=0.6533

  [CROSS-DOMAIN] Train: A_Daraz_Ecommerce (8000) -> Test: D_Electronics_Gaming_Delivery (5618)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 13131.78it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ra

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.489990,0.335906,0.850394,0.901250
2,0.327652,0.293597,0.850070,0.902500
3,0.291562,0.291989,0.875849,0.920000
4,0.302158,0.315084,0.877751,0.920000
5,0.174954,0.348277,0.882201,0.922500


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.81it/s]


  macro-F1=0.4124  accuracy=0.5139

Source A done.


In [6]:
print('=== XLM-R | Source: B_Restaurant_Food ===')

src_name, train_df, _ = domains[1]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nSource B done.')

=== XLM-R | Source: B_Restaurant_Food ===

  [CROSS-DOMAIN] Train: B_Restaurant_Food (600) -> Test: A_Daraz_Ecommerce (3398)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 15154.11it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ra

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,No log,0.839952,0.278317,0.716667
2,0.766658,0.670447,0.278317,0.716667
3,0.541409,0.638597,0.278317,0.716667


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.91it/s]


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co

...

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14072.21it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ra

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,No log,0.902809,0.278317,0.716667
2,0.813774,0.685875,0.278317,0.716667
3,0.594749,0.730954,0.278317,0.716667


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co

  macro-F1=0.1912  accuracy=0.4023

Source B done.


In [7]:
print('=== XLM-R | Source: D_Electronics_Gaming_Delivery ===')

src_name, train_df, _ = domains[2]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(XLM_MODEL_ID, 'XLM-R_Base', src_name, train_df, tgt_name, test_df)
    xlmr_results[(src_name, tgt_name)] = f1

print('\nXLM-R -- all 9 runs complete.')
print('Results so far:')
for (s, t), f1 in sorted(xlmr_results.items()):
    tag = 'IN ' if s == t else 'X  '
    print(f'  [{tag}] {s} -> {t}: {f1:.4f}')

=== XLM-R | Source: D_Electronics_Gaming_Delivery ===

  [CROSS-DOMAIN] Train: D_Electronics_Gaming_Delivery (8000) -> Test: A_Daraz_Ecommerce (3398)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16406.45it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ra

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.932293,0.965249,0.523041,0.557500
2,0.851219,0.857907,0.551503,0.607500
3,0.763554,0.823857,0.631803,0.643750
4,0.644131,0.907728,0.610581,0.646250
5,0.563891,0.890041,0.630120,0.655000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.82it/s]


  macro-F1=0.4081  accuracy=0.6368

  [CROSS-DOMAIN] Train: D_Electronics_Gaming_Delivery (8000) -> Test: B_Restaurant_Food (150)


...

  macro-F1=0.3706  accuracy=0.7867

  [IN-DOMAIN] Train: D_Electronics_Gaming_Delivery (8000) -> Test: D_Electronics_Gaming_Delivery (5618)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 13630.45it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ra

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.961963,0.966171,0.512708,0.548750
2,0.897948,0.891500,0.525818,0.601250
3,0.773100,0.855148,0.593030,0.607500
4,0.635366,0.961576,0.602193,0.640000
5,0.612179,0.890967,0.612535,0.636250


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]


  macro-F1=0.6554  accuracy=0.6757

XLM-R -- all 9 runs complete.
Results so far:
  [IN ] A_Daraz_Ecommerce -> A_Daraz_Ecommerce: 0.8631
  [X  ] A_Daraz_Ecommerce -> B_Restaurant_Food: 0.5191
  [X  ] A_Daraz_Ecommerce -> D_Electronics_Gaming_Delivery: 0.4124
  [X  ] B_Restaurant_Food -> A_Daraz_Ecommerce: 0.2496
  [IN ] B_Restaurant_Food -> B_Restaurant_Food: 0.2935
  [X  ] B_Restaurant_Food -> D_Electronics_Gaming_Delivery: 0.1912
  [X  ] D_Electronics_Gaming_Delivery -> A_Daraz_Ecommerce: 0.4081
  [X  ] D_Electronics_Gaming_Delivery -> B_Restaurant_Food: 0.3706
  [IN ] D_Electronics_Gaming_Delivery -> D_Electronics_Gaming_Delivery: 0.6554


## Model 2 — mBERT

In [8]:
print('=== mBERT | Source: A_Daraz_Ecommerce ===')
mbert_results = globals().get('mbert_results', {})

src_name, train_df, _ = domains[0]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource A done.')

=== mBERT | Source: A_Daraz_Ecommerce ===

  [IN-DOMAIN] Train: A_Daraz_Ecommerce (8000) -> Test: A_Daraz_Ecommerce (3398)


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maryam\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12437.85it/s]
[tr

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.415344,0.325096,0.835161,0.896250
2,0.272956,0.292251,0.852273,0.901250
3,0.216858,0.347589,0.849231,0.905000
4,0.175138,0.422968,0.843445,0.900000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.94it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

  macro-F1=0.8493  accuracy=0.8911

  [CROSS-DOMAIN] Train: A_Daraz_Ecommerce (8000) -> Test: B_Restaurant_Food (150)


...

  macro-F1=0.5295  accuracy=0.6600

  [CROSS-DOMAIN] Train: A_Daraz_Ecommerce (8000) -> Test: D_Electronics_Gaming_Delivery (5618)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16583.55it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.415344,0.325096,0.835161,0.896250
2,0.272956,0.292251,0.852273,0.901250
3,0.216858,0.347589,0.849231,0.905000
4,0.175138,0.422968,0.843445,0.900000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.90it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

  macro-F1=0.3106  accuracy=0.4389

Source A done.


In [9]:
print('=== mBERT | Source: B_Restaurant_Food ===')

src_name, train_df, _ = domains[1]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nSource B done.')

=== mBERT | Source: B_Restaurant_Food ===

  [CROSS-DOMAIN] Train: B_Restaurant_Food (600) -> Test: A_Daraz_Ecommerce (3398)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12437.47it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,No log,0.827180,0.278317,0.716667
2,0.742497,0.654514,0.357902,0.750000
3,0.478813,0.721678,0.357902,0.750000
4,0.478813,0.712989,0.389562,0.766667
5,0.322628,0.770758,0.409872,0.766667


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.38it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co

...

  macro-F1=0.3876  accuracy=0.7200

  [CROSS-DOMAIN] Train: B_Restaurant_Food (600) -> Test: D_Electronics_Gaming_Delivery (5618)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14215.07it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,No log,0.682314,0.278317,0.716667
2,0.764327,0.628585,0.391930,0.766667
3,0.455629,0.594511,0.473394,0.800000
4,0.455629,0.823543,0.357902,0.750000
5,0.329701,0.720617,0.447048,0.800000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.14it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

  macro-F1=0.2512  accuracy=0.4023

Source B done.


In [10]:
print('=== mBERT | Source: D_Electronics_Gaming_Delivery ===')

src_name, train_df, _ = domains[2]
for tgt_name, _, test_df in domains:
    f1, _ = run_one(BERT_MODEL_ID, 'mBERT', src_name, train_df, tgt_name, test_df)
    mbert_results[(src_name, tgt_name)] = f1

print('\nmBERT -- all 9 runs complete.')
print('Results so far:')
for (s, t), f1 in sorted(mbert_results.items()):
    tag = 'IN ' if s == t else 'X  '
    print(f'  [{tag}] {s} -> {t}: {f1:.4f}')

=== mBERT | Source: D_Electronics_Gaming_Delivery ===

  [CROSS-DOMAIN] Train: D_Electronics_Gaming_Delivery (8000) -> Test: A_Daraz_Ecommerce (3398)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13267.00it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.871269,0.919213,0.510628,0.560000
2,0.773961,0.838140,0.563970,0.597500
3,0.611320,0.909219,0.631047,0.641250
4,0.445259,1.044806,0.601073,0.632500
5,0.288605,1.122280,0.614024,0.636250


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

  macro-F1=0.4540  accuracy=0.5424

  [CROSS-DOMAIN] Train: D_Electronics_Gaming_Delivery (8000) -> Test: B_Restaurant_Food (150)


...

  macro-F1=0.4177  accuracy=0.6200

  [IN-DOMAIN] Train: D_Electronics_Gaming_Delivery (8000) -> Test: D_Electronics_Gaming_Delivery (5618)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24880.51it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.871349,0.919185,0.510628,0.560000
2,0.773881,0.837717,0.562958,0.596250
3,0.611136,0.907771,0.635607,0.645000
4,0.444287,1.048480,0.601073,0.632500
5,0.288961,1.121322,0.614207,0.636250


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.94it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

  macro-F1=0.6598  accuracy=0.6689

mBERT -- all 9 runs complete.
Results so far:
  [IN ] A_Daraz_Ecommerce -> A_Daraz_Ecommerce: 0.8493
  [X  ] A_Daraz_Ecommerce -> B_Restaurant_Food: 0.5295
  [X  ] A_Daraz_Ecommerce -> D_Electronics_Gaming_Delivery: 0.3106
  [X  ] B_Restaurant_Food -> A_Daraz_Ecommerce: 0.3814
  [IN ] B_Restaurant_Food -> B_Restaurant_Food: 0.3876
  [X  ] B_Restaurant_Food -> D_Electronics_Gaming_Delivery: 0.2512
  [X  ] D_Electronics_Gaming_Delivery -> A_Daraz_Ecommerce: 0.4540
  [X  ] D_Electronics_Gaming_Delivery -> B_Restaurant_Food: 0.4177
  [IN ] D_Electronics_Gaming_Delivery -> D_Electronics_Gaming_Delivery: 0.6598


## Table 4 — Compute and Save Results

In [11]:
def compute_table4(results, model_label):
    names = [d[0] for d in domains]
    ss = [results[(d, d)] for d in names]
    st = [results[(s, t)] for s in names for t in names if s != t]
    sd_pairs = [results[(s,s)] - results[(s,t)] for s in names for t in names if s!=t]
    td_pairs = [results[(t,t)] - results[(s,t)] for s in names for t in names if s!=t]

    row = {
        'model':  model_label,
        'task':   'T6_Roman_ProductReviews',
        'avg_SS': round(np.mean(ss), 4),
        'avg_ST': round(np.mean(st), 4),
        'avg_SD': round(np.mean(sd_pairs), 4),
        'WSD':    round(max(sd_pairs), 4),
        'avg_TD': round(np.mean(td_pairs), 4),
        'WTD':    round(max(td_pairs), 4),
    }
    print(f"\n{model_label}")
    print(f"  Avg In-Domain  (SS) : {row['avg_SS']}")
    print(f"  Avg Cross-Domain(ST): {row['avg_ST']}")
    print(f"  Avg Source Drop (SD): {row['avg_SD']}")
    print(f"  Worst Source Drop   : {row['WSD']}")
    print(f"  Avg Target Drop (TD): {row['avg_TD']}")
    print(f"  Worst Target Drop   : {row['WTD']}")
    return row

row_xlmr  = compute_table4(xlmr_results,  'XLM-R_Base')
row_mbert = compute_table4(mbert_results, 'mBERT')

summary_df = pd.DataFrame([row_xlmr, row_mbert])
print('\n-- Summary --')
print(summary_df.to_string(index=False))


XLM-R_Base
  Avg In-Domain  (SS) : 0.604
  Avg Cross-Domain(ST): 0.3585
  Avg Source Drop (SD): 0.2455
  Worst Source Drop   : 0.4507
  Avg Target Drop (TD): 0.2455
  Worst Target Drop   : 0.6135

mBERT
  Avg In-Domain  (SS) : 0.6322
  Avg Cross-Domain(ST): 0.3907
  Avg Source Drop (SD): 0.2415
  Worst Source Drop   : 0.5387
  Avg Target Drop (TD): 0.2415
  Worst Target Drop   : 0.4679

-- Summary --
     model                    task  avg_SS  avg_ST  avg_SD    WSD  avg_TD    WTD
XLM-R_Base T6_Roman_ProductReviews  0.6040  0.3585  0.2455 0.4507  0.2455 0.6135
     mBERT T6_Roman_ProductReviews  0.6322  0.3907  0.2415 0.5387  0.2415 0.4679


In [12]:
out_path = os.path.join(ROOT, 'results', 'Table4_T6_Roman_ProductReviews.csv')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
summary_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

Saved: c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\results\Table4_T6_Roman_ProductReviews.csv


---
## Table 5 — Few-Shot LLM Experiments (Robustness)

Using 5 HuggingFace models on GPU with 4-bit quantization:
- **Qwen2.5-7B-Instruct** — Best multilingual support including Urdu
- **Llama-3.1-8B-Instruct** — Strong general-purpose multilingual
- **Gemma-2-9B-it** — Google's instruction-tuned model
- **Aya-Expanse-8B** — Cohere's model built for 23 languages incl. Urdu
- **Mistral-Nemo-Instruct-2407** — 12B joint Mistral+NVIDIA multilingual

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import gc
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"



LLM_MODELS = {
    'Qwen2.5-7B':      'Qwen/Qwen2.5-7B-Instruct',
    'Llama3.1-8B':     'meta-llama/Llama-3.1-8B-Instruct',
    'Mistral-7B':      'mistralai/Mistral-7B-Instruct-v0.3',
}

K_SHOT = 5
LLM_RESULTS_BASE = os.path.join(ROOT, 'results', 'T6_Roman_ProductReviews')

print('LLM config ready.')
print(f'Models: {list(LLM_MODELS.keys())}')
print(f'K-shot: {K_SHOT} examples per class')
print('Note: Only Mistral-7B and meta-llama-8B included for focused testing')

LLM config ready.
Models: ['Qwen2.5-7B', 'Llama3.1-8B', 'Mistral-7B']
K-shot: 5 examples per class
Note: Only Mistral-7B and meta-llama-8B included for focused testing


In [14]:
MAX_TEST_LLM = 100
LLM_BATCH_SIZE = 16

LABEL_NAMES = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

def build_prompt(train_df, test_text, k=K_SHOT):
    examples = []
    for label_val in sorted(train_df['label'].unique()):
        subset = train_df[train_df['label'] == label_val]
        sampled = subset.sample(n=min(k, len(subset)), random_state=42)
        for _, row in sampled.iterrows():
            label_str = LABEL_NAMES[row['label']]
            examples.append(f"Text: {row['text']}\nLabel: {label_str}")

    prompt = (
        "You are a product review sentiment classifier for Roman Urdu text. "
        "Classify each review as 'Positive', 'Negative', or 'Neutral'. "
        "Respond with ONLY the label, nothing else.\n\n"
        "Examples:\n" + "\n\n".join(examples) + "\n\n"
        f"Text: {test_text}\nLabel:"
    )
    return prompt


def parse_llm_label(response):
    response = response.strip().lower()
    if 'positive' in response:
        return 2
    elif 'neutral' in response:
        return 1
    elif 'negative' in response:
        return 0
    elif '2' in response:
        return 2
    elif '1' in response:
        return 1
    elif '0' in response:
        return 0
    return -1


def run_llm_experiment(model_name, model_id, src_name, train_df, tgt_name, test_df, tokenizer, model):
    out_dir = os.path.join(LLM_RESULTS_BASE, model_name)
    os.makedirs(out_dir, exist_ok=True)
    result_path = os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')

    if os.path.exists(result_path):
        print(f'  [SKIP] {src_name} -> {tgt_name} already done.')
        with open(result_path, encoding='utf-8') as f:
            return json.load(f)['macro_f1']

    run_type = 'IN-DOMAIN' if src_name == tgt_name else 'CROSS-DOMAIN'

    if len(test_df) > MAX_TEST_LLM:
        n_per_class = MAX_TEST_LLM // test_df['label'].nunique()
        test_eval = pd.concat([
            group.sample(n=min(len(group), n_per_class), random_state=42)
            for _, group in test_df.groupby('label')
        ]).reset_index(drop=True)
    else:
        test_eval = test_df

    print(f'  [{run_type}] {src_name} -> {tgt_name} ({len(test_eval)} samples)')

    true_labels = test_eval['label'].astype(int).tolist()
    all_prompts = [build_prompt(train_df, row['text']) for _, row in test_eval.iterrows()]

    preds = []
    for i in range(0, len(all_prompts), LLM_BATCH_SIZE):
        batch_prompts = all_prompts[i:i+LLM_BATCH_SIZE]
        inputs = tokenizer(
            batch_prompts, return_tensors='pt', truncation=True,
            max_length=1024, padding=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=10, do_sample=False,
                temperature=1.0, pad_token_id=tokenizer.eos_token_id
            )

        for j, output in enumerate(outputs):
            input_len = inputs['input_ids'][j].ne(tokenizer.pad_token_id).sum()
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred = parse_llm_label(generated)
            if pred == -1:
                pred = 1
            preds.append(pred)

    macro_f1 = f1_score(true_labels, preds, average='macro')
    accuracy = accuracy_score(true_labels, preds)

    result = {
        'task': 'T6_Roman_ProductReviews', 'model': model_name, 'model_id': model_id,
        'source': src_name, 'target': tgt_name, 'type': run_type,
        'k_shot': K_SHOT, 'test_size': len(test_eval),
        'macro_f1': round(macro_f1, 4), 'accuracy': round(accuracy, 4),
        'classification_report': classification_report(true_labels, preds, output_dict=True, zero_division=0)
    }
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f'    F1={macro_f1:.4f}  Acc={accuracy:.4f}')
    return macro_f1


print('LLM helpers loaded (batched inference, max test =', MAX_TEST_LLM, ')')

LLM helpers loaded (batched inference, max test = 100 )


In [15]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

all_llm_results = {}

for model_name, model_id in LLM_MODELS.items():
    out_dir = os.path.join(LLM_RESULTS_BASE, model_name)
    all_done = True
    for src_name, _, _ in domains:
        for tgt_name, _, _ in domains:
            if not os.path.exists(os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')):
                all_done = False
                break
        if not all_done:
            break

    if all_done:
        print(f'\n[SKIP ALL] {model_name} — all 9 results already exist.')
        llm_results = {}
        for src_name, _, _ in domains:
            for tgt_name, _, _ in domains:
                with open(os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json'), encoding='utf-8') as f:
                    llm_results[(src_name, tgt_name)] = json.load(f)['macro_f1']
        all_llm_results[model_name] = llm_results
        continue

    print(f'\n{"="*60}')
    print(f'Loading {model_name} ({model_id})...')
    print(f'{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map='auto',
        token=HF_TOKEN,
        disable_mmap=True
    )
    model.eval()
    print(f'{model_name} loaded successfully.')

    llm_results = {}
    for src_name, train_df, _ in domains:
        for tgt_name, _, test_df in domains:
            f1 = run_llm_experiment(model_name, model_id, src_name, train_df, tgt_name, test_df, tokenizer, model)
            llm_results[(src_name, tgt_name)] = f1

    all_llm_results[model_name] = llm_results

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\n{model_name} -- all 9 runs complete. Memory freed.')


Loading Qwen2.5-7B (Qwen/Qwen2.5-7B-Instruct)...


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maryam\.cache\huggingface\hub\models--Qwen--Qwen2.5-7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 339/339 [00:01<00:00, 185.29it/s]


Qwen2.5-7B loaded successfully.
  [IN-DOMAIN] A_Daraz_Ecommerce -> A_Daraz_Ecommerce (99 samples)
    F1=0.3436  Acc=0.4343
  [CROSS-DOMAIN] A_Daraz_Ecommerce -> B_Restaurant_Food (65 samples)
    F1=0.4554  Acc=0.5692
  [CROSS-DOMAIN] A_Daraz_Ecommerce -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2728  Acc=0.3535
  [CROSS-DOMAIN] B_Restaurant_Food -> A_Daraz_Ecommerce (99 samples)
    F1=0.3520  Acc=0.4343
  [IN-DOMAIN] B_Restaurant_Food -> B_Restaurant_Food (65 samples)
    F1=0.4965  Acc=0.6000
  [CROSS-DOMAIN] B_Restaurant_Food -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2805  Acc=0.3636
  [CROSS-DOMAIN] D_Electronics_Gaming_Delivery -> A_Daraz_Ecommerce (99 samples)
    F1=0.3301  Acc=0.4242
  [CROSS-DOMAIN] D_Electronics_Gaming_Delivery -> B_Restaurant_Food (65 samples)
    F1=0.3992  Acc=0.5385
  [IN-DOMAIN] D_Electronics_Gaming_Delivery -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2391  Acc=0.3333

Qwen2.5-7B -- all 9 runs complete. Memory freed.


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maryam\.cache\huggingface\hub\models--meta-llama--Llama-3.1-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 164.28it/s]


Llama3.1-8B loaded successfully.
  [IN-DOMAIN] A_Daraz_Ecommerce -> A_Daraz_Ecommerce (99 samples)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


    F1=0.3269  Acc=0.4242
  [CROSS-DOMAIN] A_Daraz_Ecommerce -> B_Restaurant_Food (65 samples)
    F1=0.4772  Acc=0.6000
  [CROSS-DOMAIN] A_Daraz_Ecommerce -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2766  Acc=0.3636
  [CROSS-DOMAIN] B_Restaurant_Food -> A_Daraz_Ecommerce (99 samples)
    F1=0.3255  Acc=0.4242
  [IN-DOMAIN] B_Restaurant_Food -> B_Restaurant_Food (65 samples)
    F1=0.5000  Acc=0.6154
  [CROSS-DOMAIN] B_Restaurant_Food -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2342  Acc=0.3434
  [CROSS-DOMAIN] D_Electronics_Gaming_Delivery -> A_Daraz_Ecommerce (99 samples)
    F1=0.3058  Acc=0.4141
  [CROSS-DOMAIN] D_Electronics_Gaming_Delivery -> B_Restaurant_Food (65 samples)
    F1=0.4855  Acc=0.6154
  [IN-DOMAIN] D_Electronics_Gaming_Delivery -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2073  Acc=0.3535

Llama3.1-8B -- all 9 runs complete. Memory freed.

Loading Mistral-7B (mistralai/Mistral-7B-Instruct-v0.3)...


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maryam\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 179.67it/s

Mistral-7B loaded successfully.
  [IN-DOMAIN] A_Daraz_Ecommerce -> A_Daraz_Ecommerce (99 samples)
    F1=0.4471  Acc=0.4949
  [CROSS-DOMAIN] A_Daraz_Ecommerce -> B_Restaurant_Food (65 samples)
    F1=0.5601  Acc=0.6000
  [CROSS-DOMAIN] A_Daraz_Ecommerce -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2648  Acc=0.3434
  [CROSS-DOMAIN] B_Restaurant_Food -> A_Daraz_Ecommerce (99 samples)
    F1=0.4326  Acc=0.4747
  [IN-DOMAIN] B_Restaurant_Food -> B_Restaurant_Food (65 samples)
    F1=0.5540  Acc=0.6154
  [CROSS-DOMAIN] B_Restaurant_Food -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2116  Acc=0.3131
  [CROSS-DOMAIN] D_Electronics_Gaming_Delivery -> A_Daraz_Ecommerce (99 samples)
    F1=0.3633  Acc=0.4444
  [CROSS-DOMAIN] D_Electronics_Gaming_Delivery -> B_Restaurant_Food (65 samples)
    F1=0.4934  Acc=0.5846
  [IN-DOMAIN] D_Electronics_Gaming_Delivery -> D_Electronics_Gaming_Delivery (99 samples)
    F1=0.2391  Acc=0.3333

Mistral-7B -- all 9 runs complete. Memory freed.


## Table 5 — Compute and Save Results

In [16]:
def compute_table5_row(results, model_label):
    names = [d[0] for d in domains]
    st = [results[(s, t)] for s in names for t in names if s != t]
    ss = [results[(d, d)] for d in names]
    return {
        'model': model_label,
        'avg_SS': round(np.mean(ss), 4),
        'avg_ST': round(np.mean(st), 4),
    }

table5_rows = []
table5_rows.append({**compute_table5_row(xlmr_results, 'XLM-R_Base (fine-tuned)'), 'type': 'fine-tuned'})
table5_rows.append({**compute_table5_row(mbert_results, 'mBERT (fine-tuned)'), 'type': 'fine-tuned'})

for model_name, results in all_llm_results.items():
    table5_rows.append({**compute_table5_row(results, f'{model_name} (5-shot)'), 'type': 'few-shot'})

table5_df = pd.DataFrame(table5_rows)
print('\n=== TABLE 5: Robustness Comparison (Product Reviews) ===')
print(table5_df.to_string(index=False))

out_path = os.path.join(ROOT, 'results', 'Table5_T6_Roman_ProductReviews.csv')
table5_df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')


=== TABLE 5: Robustness Comparison (Product Reviews) ===
                  model  avg_SS  avg_ST       type
XLM-R_Base (fine-tuned)  0.6040  0.3585 fine-tuned
     mBERT (fine-tuned)  0.6322  0.3907 fine-tuned
    Qwen2.5-7B (5-shot)  0.3597  0.3483   few-shot
   Llama3.1-8B (5-shot)  0.3448  0.3508   few-shot
    Mistral-7B (5-shot)  0.4134  0.3876   few-shot

Saved: c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\results\Table5_T6_Roman_ProductReviews.csv


---
## HuggingFace Upload — Open Source Contribution

In [17]:
from huggingface_hub import HfApi, login

login(token=HF_TOKEN)
print('Logged in to HuggingFace.')

Logged in to HuggingFace.
